<div align="center">

# Universidad de Sevilla  
## Grado en Ingeniería del Software  
### Escuela Técnica Superior de Ingeniería Informática  
### Inteligencia Artificial (IA) – Curso 2025/2026  

<img src="../../img/portada.png" width="300"/>

---

#  Aprendizaje automático relacional  
### Primera Convocatoria – Junio 2026  

**Profesor:** Pedro Almagro Blanco

**Autores:**  
David Espina Apellaniz  
Marco Padilla Gómez  

**Fecha de entrega:** 8 de Junio de 2025  

</div>

# 02 - Extracción de features relacionales

En este notebook se transforma el grafo de Twitch en una tabla apta para entrenar modelos de Machine Learning.

El objetivo es generar el archivo:

`data/processed/twitch_mature_features.csv`

Este archivo será usado por todos los notebooks de modelos.


## 1. Importación de librerías y configuración inicial

In [1]:
import sys
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

# Permite importar módulos desde la carpeta src cuando el notebook está dentro de subcarpetas
sys.path.append("../../")

from src.data_loader import load_edges, load_target, load_json_features
from src.graph_features import build_graph, get_graph_summary, compute_graph_features
from src.preprocessing import (
    json_features_to_dataframe,
    build_final_dataset,
    save_processed_dataset
)
from src.visualization import save_correlation_matrix

DATA_RAW_PATH = Path("../../data/raw")
DATA_PROCESSED_PATH = Path("../../data/processed")
IMG_PATH = Path("../../img")

DATA_PROCESSED_PATH.mkdir(parents=True, exist_ok=True)
IMG_PATH.mkdir(parents=True, exist_ok=True)


## 2. Carga de los datos originales

In [2]:
edges = load_edges(DATA_RAW_PATH / "ES_edges.csv")
target = load_target(DATA_RAW_PATH / "ES_target.csv")
features_json = load_json_features(DATA_RAW_PATH / "ES_features.json")

print("Edges shape:", edges.shape)
print("Target shape:", target.shape)
print("Número de nodos en JSON:", len(features_json))


Edges shape: (59382, 2)
Target shape: (4648, 6)
Número de nodos en JSON: 4648


## 3. Construcción del grafo

In [3]:
graph = build_graph(edges)
summary = get_graph_summary(graph)

summary

{'num_nodes': 4648,
 'num_edges': 59382,
 'density': 0.005498522726893927,
 'is_connected': True,
 'num_connected_components': 1,
 'largest_component_size': 4648}

## 4. Cálculo de métricas relacionales

Se calculan las principales métricas relacionales de cada nodo:

- `degree`
- `degree_centrality`
- `clustering`
- `pagerank`
- `closeness`
- `betweenness`
- `community`

La métrica `betweenness` se calcula de forma aproximada para reducir el tiempo de ejecución.


In [ ]:
graph_features = compute_graph_features(
    graph,
    betweenness_k=500,
    random_state=42
)

graph_features.head()

In [ ]:
graph_features.describe()

## 5. Transformación del JSON en variable `num_features`

El archivo JSON contiene listas de características para cada usuario. Para este primer enfoque se crea una variable simple:

`num_features = número de características asociadas a cada usuario`


In [ ]:
json_features_df = json_features_to_dataframe(features_json)
json_features_df.head()

In [ ]:
json_features_df["num_features"].describe()

## 6. Unión de métricas relacionales, JSON y variable objetivo

Se construye el dataset final combinando:

- métricas relacionales del grafo;
- número de características del JSON;
- variables `days` y `views`;
- variable objetivo `mature`.


In [ ]:
final_df = build_final_dataset(
    graph_features=graph_features,
    target=target,
    json_features=json_features_df
)

final_df.head()

In [ ]:
final_df.info()

In [ ]:
final_df["mature"].value_counts()

## 7. Comprobación de valores nulos

In [ ]:
final_df.isnull().sum()

## 8. Matriz de correlación

Se genera una matriz de correlación para analizar la relación entre las variables numéricas y detectar posibles relaciones entre métricas.


In [ ]:
save_correlation_matrix(
    final_df,
    IMG_PATH / "matriz_correlacion.png"
)

# Mostrar también en el notebook
corr = final_df.corr(numeric_only=True)

plt.figure(figsize=(10, 8))
plt.imshow(corr, aspect="auto")
plt.colorbar()
plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
plt.yticks(range(len(corr.columns)), corr.columns)
plt.title("Matriz de correlación")
plt.tight_layout()
plt.show()


## 9. Guardado del dataset procesado

Este CSV será el punto de partida para todos los modelos:

- Decision Tree
- Random Forest
- kNN
- Logistic Regression


In [ ]:
output_path = DATA_PROCESSED_PATH / "twitch_mature_features.csv"

save_processed_dataset(final_df, output_path)

print(f"Dataset procesado guardado en: {output_path}")


## 10. Comprobación final

Se carga de nuevo el CSV generado para comprobar que se ha guardado correctamente.


In [ ]:
processed_df = pd.read_csv(DATA_PROCESSED_PATH / "twitch_mature_features.csv")

processed_df.head()

In [ ]:
processed_df.shape

## 11. Conclusiones

Conclusiones que se pueden usar en la memoria:

- Se ha generado una tabla final con una fila por usuario/nodo.
- Cada nodo queda representado mediante métricas relacionales calculadas a partir del grafo.
- La variable objetivo es `mature`.
- El archivo `twitch_mature_features.csv` permite entrenar modelos clásicos de Machine Learning para resolver una tarea de clasificación relacional de nodos.
